# Merge tables without double counting
Python practice from [The Joinery](https://thejoinery.dev/). Run each cell with **Shift + Enter**, from the top. The explanations are on the guide page: https://thejoinery.dev/#/projects/python-merge

## 1. Read the three tables

In [ ]:
import pandas as pd

BASE = "https://thejoinery.dev/projects/python-data/"
headers = {"User-Agent": "Mozilla/5.0"}   # some websites block Python's default request; this sends a browser-style name
orders = pd.read_csv(BASE + "orders.csv", storage_options=headers)
customers = pd.read_csv(BASE + "customers.csv", storage_options=headers)
items = pd.read_csv(BASE + "order_items.csv", storage_options=headers)
print("orders", orders.shape)
print("customers", customers.shape)
print("items", items.shape)

## 2. Check the keys first

In [ ]:
print(orders["order_id"].is_unique)
print(customers["customer_id"].is_unique)
print(items["order_id"].is_unique)
print(items["order_id"].value_counts().head(3))

## 3. Left merge: keep every order

In [ ]:
with_customers = orders.merge(customers, on="customer_id", how="left", indicator=True)
print(with_customers.shape)
print(with_customers["_merge"].value_counts())

## 4. Inner merge: only the matches

In [ ]:
inner = orders.merge(customers, on="customer_id", how="inner")
print(len(orders), len(with_customers), len(inner))

## 5. Watch the totals grow

In [ ]:
print(round(orders["order_total"].sum(), 2))
joined = orders.merge(items, on="order_id", how="left")
print(joined.shape)
print(round(joined["order_total"].sum(), 2))

## 6. Fix it: summarise first, then merge

In [ ]:
per_order = (items
             .groupby("order_id")
             .agg(items=("item_number", "count"), items_total=("price", "sum"))
             .reset_index())
orders_with_items = orders.merge(per_order, on="order_id", how="left", validate="one_to_one")
print(orders_with_items.shape)
print(round(orders_with_items["order_total"].sum(), 2))
print(orders_with_items[["order_id", "order_total", "items", "items_total"]].head())

In [ ]:
try:
    orders.merge(items, on="order_id", validate="one_to_one")
except pd.errors.MergeError as problem:
    print("Stopped:", problem)

## 7. Answer item questions from the items table

In [ ]:
by_category = items.groupby("category")["price"].sum().round(2).sort_values(ascending=False)
print(by_category)